#### MILE -- pSMILE Mini-Batch (02c), Produktionslauf mit finaler Step Size

Identischer Aufbau wie `02c_mile_psmile_test.ipynb` (der Lauf, der die bisher berichteten
pSMILE-Zahlen lieferte), mit zwei Aenderungen:

- **RUN_NAME enthaelt jetzt die Step Size**, damit mehrere Kandidaten (unterschiedliche
  Step Sizes) nicht dieselben Ergebnis-Dateien ueberschreiben.
- **Evaluation auf dem Validation-Set statt dem Testset** (`EVAL_MODE="val"`,
  `data.get("val")`, 12.000 Beispiele statt der 7.600 Test-Beispiele). Grund: die Step Size
  ist ein Hyperparameter, der hier erst noch ausgewaehlt wird -- ihn anhand von Testset-Metriken
  auszuwaehlen waere Testset-Contamination (das Testset wuerde dann nicht mehr eine unabhaengige,
  unverzerrte Abschaetzung der finalen Performance liefern). Sobald ein Kandidat anhand der
  Validation-Metriken feststeht, wird NUR fuer diesen einen finalen Lauf zusaetzlich auf dem
  Testset ausgewertet (separater Schritt, nicht Teil dieses Notebooks).


In [1]:
from pathlib import Path
import os
import json as _json

# ============================================================
# CONFIG (02c -- pSMILE/NOISYSMILE Produktionslauf, Step-Size-Kandidat)
# ============================================================

MASTER_DIR = Path("/dss/dsshome1/00/ra58vit2/Masterarbeit")
QWEN_REPO = MASTER_DIR / "bayes_sub_inf"
SMILE_REPO = MASTER_DIR / "SMILE"

BASELINE_SCRIPT = (
    QWEN_REPO / "experiments" / "ag_news_qwen_lora" / "evaluate_saved_baseline.py"
)
RESULT_DIR = MASTER_DIR / "method_results" / "mile_qwen_agnews"
CHECKPOINTS_ROOT = Path.home() / "Masterarbeit" / "checkpoints"

CHAIN_SEEDS = _json.loads(os.environ.get("CHAIN_SEEDS", "[1,2,3]"))

SEQ_LEN = 32
PRIOR_STD = float(os.environ.get("PRIOR_STD", "0.02"))

MINIBATCH_SIZE = int(os.environ.get("MINIBATCH_SIZE", "32"))
NLL_INNER_BATCH_SIZE = 16
assert MINIBATCH_SIZE % NLL_INNER_BATCH_SIZE == 0, (
    f"MINIBATCH_SIZE ({MINIBATCH_SIZE}) muss durch NLL_INNER_BATCH_SIZE ({NLL_INNER_BATCH_SIZE}) teilbar sein."
)

# EIN fester Anfangswert fuer diesen Kandidaten-Lauf (kein Grid mehr).
STEP_SIZE_GRID = _json.loads(os.environ.get("STEP_SIZE_GRID", "[0.0001]"))
assert len(STEP_SIZE_GRID) == 1, (
    f"STEP_SIZE_GRID hat {len(STEP_SIZE_GRID)} Werte -- dieses Notebook ist fuer genau EINEN "
    "Kandidaten pro Lauf gedacht (RUN_NAME haengt direkt davon ab)."
)
INITIAL_STEP_SIZE = STEP_SIZE_GRID[0]

NUM_INTEGRATION_STEPS = int(os.environ.get("NUM_INTEGRATION_STEPS", "1"))
RESET_PROB = float(os.environ.get("RESET_PROB", "0.99"))
ADAPTION_PROB = float(os.environ.get("ADAPTION_PROB", "0.05"))
ADAPTION_STRENGTH = float(os.environ.get("ADAPTION_STRENGTH", "0.02"))
SMOOTHING_FACTOR = float(os.environ.get("SMOOTHING_FACTOR", "0.01"))

N_SAMPLES_PER_CHAIN = int(os.environ.get("N_SAMPLES_PER_CHAIN", "90"))
THINNING_STEPS = int(os.environ.get("THINNING_STEPS", "30"))
BURN_IN_SAMPLES = int(os.environ.get("BURN_IN_SAMPLES", "20"))

EVAL_BATCH_SIZE = 64
# Validation, NICHT Test -- Stepsize-Auswahl darf das Testset nicht anfassen
# (siehe Intro-Zelle). Fuer den finalen, ausgewaehlten Kandidaten wird
# separat noch einmal auf "test" ausgewertet.
EVAL_MODE = os.environ.get("EVAL_MODE", "val")

N_CHAINS = len(CHAIN_SEEDS)

RUN_NAME = (
    f"mile_psmile_k{N_CHAINS}"
    f"_prior{PRIOR_STD}"
    f"_step{INITIAL_STEP_SIZE}"
    f"_mb{MINIBATCH_SIZE}"
    f"_spc{N_SAMPLES_PER_CHAIN}"
    f"_thin{THINNING_STEPS}"
    f"_{EVAL_MODE}eval"
)

print("======================================")
print("MILE -- pSMILE/NOISYSMILE PRODUKTIONSLAUF (02c, Step-Size-Kandidat)")
print("======================================")
print("Run name:              ", RUN_NAME)
print("Chain seeds:           ", CHAIN_SEEDS)
print("Prior std:             ", PRIOR_STD)
print("Minibatch size:        ", MINIBATCH_SIZE)
print("Initial step size:     ", INITIAL_STEP_SIZE)
print("num_integration_steps: ", NUM_INTEGRATION_STEPS)
print("reset_prob:            ", RESET_PROB, " (Library-Default)")
print("adaption_prob:         ", ADAPTION_PROB, " (Library-Default)")
print("adaption_strength:     ", ADAPTION_STRENGTH, " (Library-Default)")
print("smoothing_factor:      ", SMOOTHING_FACTOR, " (Library-Default)")
print("N_SAMPLES_PER_CHAIN:   ", N_SAMPLES_PER_CHAIN)
print("THINNING_STEPS:        ", THINNING_STEPS)
print("BURN_IN_SAMPLES:       ", BURN_IN_SAMPLES)
print("EVAL_MODE:             ", EVAL_MODE)
print("======================================")


MILE -- pSMILE/NOISYSMILE PRODUKTIONSLAUF (02c, Step-Size-Kandidat)
Run name:               mile_psmile_k3_prior0.02_step0.0005_mb32_spc90_thin30_valeval
Chain seeds:            [1, 2, 3]
Prior std:              0.02
Minibatch size:         32
Initial step size:      0.0005
num_integration_steps:  1
reset_prob:             0.99  (Library-Default)
adaption_prob:          0.05  (Library-Default)
adaption_strength:      0.02  (Library-Default)
smoothing_factor:       0.01  (Library-Default)
N_SAMPLES_PER_CHAIN:    90
THINNING_STEPS:         30
BURN_IN_SAMPLES:        20
EVAL_MODE:              val


In [2]:
# ============================================================
# SMILE-REPO KLONEN (falls noch nicht vorhanden) + IMPORTS
# ============================================================

import sys
import subprocess
import runpy
import time

import numpy as np
import jax
import jax.numpy as jnp
from jax import random

if not SMILE_REPO.exists():
    print(f"SMILE-Repo nicht gefunden unter {SMILE_REPO} -- klone es jetzt...")
    result = subprocess.run(
        ["git", "clone", "https://github.com/EmanuelSommer/SMILE", str(SMILE_REPO)],
        capture_output=True, text=True,
    )
    print(result.stdout[-1500:])
    if result.returncode != 0:
        print(result.stderr[-1500:])
        raise RuntimeError("git clone von SMILE fehlgeschlagen -- siehe Fehlermeldung oben.")
    print("SMILE-Repo geklont:", SMILE_REPO)
else:
    print("SMILE-Repo bereits vorhanden:", SMILE_REPO)

os.chdir(QWEN_REPO)
if str(QWEN_REPO) not in sys.path:
    sys.path.insert(0, str(QWEN_REPO))
if str(SMILE_REPO) not in sys.path:
    sys.path.insert(0, str(SMILE_REPO))

print("Python:", sys.executable)
print("JAX:", jax.__version__)
print("Devices:", jax.devices())

assert QWEN_REPO.exists()
assert BASELINE_SCRIPT.exists()
assert SMILE_REPO.exists()


SMILE-Repo bereits vorhanden: /dss/dsshome1/00/ra58vit2/Masterarbeit/SMILE
Python: /dss/dsshome1/00/ra58vit2/Masterarbeit/bayes_sub_inf/.venv/bin/python3
JAX: 0.5.3


Devices: [CudaDevice(id=0)]


In [3]:
# ============================================================
# NOISYSMILE-KERNEL AUS DEM SMILE-REPO IMPORTIEREN ("pSMILE")
# ============================================================

try:
    from src.kernels.smile import NOISYSMILE
except ImportError as exc:
    raise ImportError(
        "Konnte 'src.kernels.smile.NOISYSMILE' nicht importieren. Pruefe, ob SMILE_REPO "
        "korrekt geklont wurde und die src/-Struktur wie im GitHub-Repo aussieht. "
        f"Original-Fehler: {exc}"
    )

print("NOISYSMILE-Kernel (pSMILE) erfolgreich importiert:", NOISYSMILE)


NOISYSMILE-Kernel (pSMILE) erfolgreich importiert: <class 'src.kernels.smile.NOISYSMILE'>


In [4]:
# ============================================================
# GETEILTES MODUL
# ============================================================

import qwen_posterior_utils as qpu

qpu.patch_subspace_curve_predict()


Patched SubspaceBaseModel._predict: lax.cond bypass for concrete train flags active.


In [5]:
# ============================================================
# LOAD QWEN AG-NEWS BASELINE (env/data + LoRA-Struktur)
# ============================================================

baseline_objects = runpy.run_path(str(BASELINE_SCRIPT))

env = baseline_objects["env"]
params = baseline_objects["params"]
data = baseline_objects["data"]
rng_key = baseline_objects["rng_key"]

print("Baseline geladen. Model:", type(env.s_model))


Dataset loaded: 120000 samples, 4 classes
Target token IDs: [[15]
 [16]
 [17]
 [18]]


Validation data are used
Training data: 108000 samples, Validation data: 12000 samples
Computed cp_fix: [1]
Setting num_epochs to 1 to match 500 steps


Using cosine rho schedule with total steps 500
save best parameters
Loaded checkpoint: tmp_files/955nvvay_pretrained_params.npy
Saved object shape: (1,)


post_predict_logits shape: (12000, 4)


post_predict_logits shape: (12000, 4)


post_predict_logits shape: (7600, 4)


post_predict_logits shape: (7600, 4)

SAVED BEST CHECKPOINT - VALIDATION
val_ll: -0.246199
val_acc: 0.913667
val_ece: 0.009229
val_brier: 0.131579
val_mutual_information: 0.000000
val_mean_entropy: 0.000000

SAVED BEST CHECKPOINT - TEST
test_ll: -0.251077
test_acc: 0.914737
test_ece: 0.007497
test_brier: 0.131565
test_mutual_information: 0.000000
test_mean_entropy: 0.000000
Baseline geladen. Model: <class 'subspace_inference.curve_optimizer.subspace_curve.LoRACategorySubspace'>


In [6]:
# ============================================================
# VOLLER TRAININGSSATZ (108.000 Beispiele) fuer Mini-Batch-Sampling
# ============================================================

train_inputs_full, train_labels_full = data.get("train")
N_TOTAL_TRAIN = int(train_labels_full.shape[0])
print("Voller Trainingssatz:", N_TOTAL_TRAIN, "Beispiele")


def sample_minibatch(rng_key):
    rng_key, batch_key = random.split(rng_key)
    batch_indices = random.randint(
        batch_key, shape=(MINIBATCH_SIZE,), minval=0, maxval=N_TOTAL_TRAIN,
    )
    x_batch = {
        key: value[batch_indices, -SEQ_LEN:]
        for key, value in train_inputs_full.items()
    }
    y_batch = train_labels_full[batch_indices]
    return x_batch, y_batch, rng_key


Voller Trainingssatz: 108000 Beispiele


In [7]:
# ============================================================
# LORA-VEKTORISIERUNG + MINI-BATCH LOG-POSTERIOR + GRAD-ESTIMATOR
# ============================================================

_, _, _, qwen_subset_nll_sum = qpu.build_qwen_nll_fns(env)

theta_map, all_leaves, lora_indices, full_tree_definition, unravel_lora = (
    qpu.vectorize_lora_params(params["params"])
)
rebuild_full_params = qpu.build_rebuild_fn(
    all_leaves, lora_indices, full_tree_definition, unravel_lora,
)

NLL_KEY_DUMMY = random.PRNGKey(0)  # lora_rho_w=lora_rho_s=0 -> key wirkt sich nicht aus


def qwen_log_posterior_minibatch(theta, x_batch, y_batch, n_total):
    candidate_params = rebuild_full_params(theta)
    batch_keys = random.split(NLL_KEY_DUMMY, y_batch.shape[0])

    nll_sum_batch, _ = qwen_subset_nll_sum(
        candidate_params, x_batch, y_batch, batch_keys,
        batch_size=NLL_INNER_BATCH_SIZE,
    )
    batch_size = y_batch.shape[0]
    nll_estimate_full = (nll_sum_batch / batch_size) * n_total

    theta_float32 = theta.astype(jnp.float32)
    squared_norm = jnp.sum(jnp.square(theta_float32))

    log_likelihood_estimate = -nll_estimate_full
    log_prior = -0.5 * squared_norm / jnp.asarray(PRIOR_STD**2, dtype=jnp.float32)

    return log_likelihood_estimate + log_prior


def grad_estimator(position, x, y):
    def scalar_fn(theta_):
        return qwen_log_posterior_minibatch(theta_, x, y, N_TOTAL_TRAIN)
    return jax.value_and_grad(scalar_fn)(position)


print("theta_map shape:", theta_map.shape)
print("grad_estimator bereit.")


LoRA objects: 48
Sampling vector shape: (540672,)
Sampling parameters: 540672
All MAP parameters finite: True
theta_map shape: (540672,)
grad_estimator bereit.


In [8]:
# ============================================================
# CHAIN-STARTPUNKTE LADEN
# ============================================================

_eval_script_template = BASELINE_SCRIPT.read_text(encoding="utf-8")


def load_chain_theta_map(seed):
    seed_dir = CHECKPOINTS_ROOT / f"qwen_lora_agnews_seed{seed}"
    checkpoint_path = list(seed_dir.glob("*_pretrained_params.npy"))[0]
    old_checkpoint_line = next(
        line for line in _eval_script_template.splitlines()
        if line.strip().startswith("CHECKPOINT_PATH")
    )
    old_seed_line = next(
        line for line in _eval_script_template.splitlines()
        if line.strip().startswith("SEED")
    )
    script_text = _eval_script_template.replace(
        old_checkpoint_line, f'CHECKPOINT_PATH = r"{checkpoint_path}"',
    ).replace(old_seed_line, f"SEED = {seed}")
    tmp_script_path = (
        QWEN_REPO / "experiments" / "ag_news_qwen_lora" / f"_tmp_eval_chain_seed{seed}_02c_prod.py"
    )
    tmp_script_path.write_text(script_text, encoding="utf-8")
    try:
        chain_objects = runpy.run_path(str(tmp_script_path))
    finally:
        tmp_script_path.unlink(missing_ok=True)
    theta, *_ = qpu.vectorize_lora_params(chain_objects["params"]["params"])
    return theta


chain_theta_maps = {seed: load_chain_theta_map(seed) for seed in CHAIN_SEEDS}
print("Chain-Startpunkte geladen:", list(chain_theta_maps.keys()))


Dataset loaded: 120000 samples, 4 classes
Target token IDs: [[15]
 [16]
 [17]
 [18]]
Validation data are used
Training data: 108000 samples, Validation data: 12000 samples
Computed cp_fix: [1]
Setting num_epochs to 1 to match 500 steps


Using cosine rho schedule with total steps 500
save best parameters
Loaded checkpoint: /dss/dsshome1/00/ra58vit2/Masterarbeit/checkpoints/qwen_lora_agnews_seed1/iiljya0g_pretrained_params.npy
Saved object shape: (1,)


post_predict_logits shape: (12000, 4)


post_predict_logits shape: (12000, 4)


post_predict_logits shape: (7600, 4)


...

post_predict_logits shape: (12000, 4)


post_predict_logits shape: (7600, 4)


post_predict_logits shape: (7600, 4)



SAVED BEST CHECKPOINT - VALIDATION
val_ll: -0.244705
val_acc: 0.916083
val_ece: 0.006208
val_brier: 0.128254
val_mutual_information: 0.000000
val_mean_entropy: 0.000000

SAVED BEST CHECKPOINT - TEST
test_ll: -0.258213
test_acc: 0.911184
test_ece: 0.009574
test_brier: 0.134704
test_mutual_information: 0.000000
test_mean_entropy: 0.000000
LoRA objects: 48
Sampling vector shape: (540672,)
Sampling parameters: 540672
All MAP parameters finite: True


Chain-Startpunkte geladen: [1, 2, 3]


In [9]:
# ============================================================
# NOISYSMILE OHNE PMAP INSTANZIIEREN (Single-GPU-Bypass)
# ============================================================

psmile_obj = object.__new__(NOISYSMILE)
psmile_obj._grad_estimator = grad_estimator

print("NOISYSMILE-Objekt (ohne pmap) bereit:", psmile_obj)


NOISYSMILE-Objekt (ohne pmap) bereit: <src.kernels.smile.NOISYSMILE object at 0x77709d21b3d0>


In [10]:
# ============================================================
# RUN_ONE_CHAIN_PSMILE -- jax.jit von Anfang an eingebaut.
# ============================================================

import functools


def build_jitted_psmile_step(num_integration_steps, reset_prob, adaption_prob,
                              adaption_strength, smoothing_factor):
    partial_step = functools.partial(
        NOISYSMILE._sample_step,
        psmile_obj,
        num_integration_steps=num_integration_steps,
        reset_prob=reset_prob,
        adaption_prob=adaption_prob,
        adaption_strength=adaption_strength,
        smoothing_factor=smoothing_factor,
    )
    return jax.jit(partial_step)


def run_one_chain_psmile(chain_theta_map, rng_key, initial_step_size, chain_label):
    # Fuenf Felder als JAX-Skalare (nicht Python float) uebergeben, damit
    # NOISYSMILE.State.__init__ sie nicht faelschlich auf "Anzahl Parameter"
    # tiled (siehe frueherer Bugfix: State ist fuer einen einzelnen flachen
    # Positions-Vektor gedacht, nicht fuer eine gepmappte Chain-Batch-Achse).
    state = NOISYSMILE.State(
        position=chain_theta_map,
        step_size=jnp.asarray(initial_step_size, dtype=jnp.float32),
        logdensity=jnp.asarray(0.0, dtype=jnp.float32),
        energy_var_mean=jnp.asarray(0.0, dtype=jnp.float32),
        energy_var_std=jnp.asarray(0.0, dtype=jnp.float32),
        time=jnp.asarray(1.0, dtype=jnp.float32),
    )
    collected_samples = []
    collected_step_sizes = []

    jitted_step = build_jitted_psmile_step(
        num_integration_steps=NUM_INTEGRATION_STEPS,
        reset_prob=RESET_PROB,
        adaption_prob=ADAPTION_PROB,
        adaption_strength=ADAPTION_STRENGTH,
        smoothing_factor=SMOOTHING_FACTOR,
    )

    for sample_index in range(N_SAMPLES_PER_CHAIN):
        sample_start_time = time.time()

        for _ in range(THINNING_STEPS):
            rng_key, step_key = random.split(rng_key)
            x_batch, y_batch, rng_key = sample_minibatch(rng_key)

            state = jitted_step(state, step_key, (x_batch, y_batch))

        sample_duration = time.time() - sample_start_time

        position_host = np.asarray(jax.device_get(state.position))
        if not np.isfinite(position_host).all():
            print(f"[{chain_label}] Sample {sample_index+1}/{N_SAMPLES_PER_CHAIN}: NICHT endlich -- abgebrochen.")
            return None, None, rng_key

        distance = float(np.linalg.norm(position_host - np.asarray(chain_theta_map)))
        mean_step_size = float(jnp.mean(state.step_size))
        print(
            f"[{chain_label}] Sample {sample_index+1}/{N_SAMPLES_PER_CHAIN}: "
            f"endlich, Distanz vom Start={distance:.4f}, mean(step_size)={mean_step_size:.6g}, "
            f"Dauer={sample_duration:.1f}s"
        )
        collected_samples.append(position_host)
        collected_step_sizes.append(mean_step_size)

    sample_positions = np.stack(collected_samples, axis=0)
    return sample_positions, collected_step_sizes, rng_key


In [11]:
# ============================================================
# MULTI-CHAIN-PRODUKTIONSLAUF -- EIN fester Anfangs-step_size-Wert
# (INITIAL_STEP_SIZE aus der Config-Zelle), alle CHAIN_SEEDS.
# ============================================================

import gc

print(f"Fester Anfangs-step_size fuer diesen Lauf: {INITIAL_STEP_SIZE}")
print(f"Chains: {CHAIN_SEEDS}   N_SAMPLES_PER_CHAIN={N_SAMPLES_PER_CHAIN}   THINNING_STEPS={THINNING_STEPS}")

chain_results = {}
chain_step_size_traces = {}

for seed in CHAIN_SEEDS:
    print(f"\n=== Chain seed={seed} ===")
    local_rng = random.PRNGKey(4242 + seed)
    samples, step_sizes, local_rng = run_one_chain_psmile(
        chain_theta_maps[seed], local_rng, INITIAL_STEP_SIZE, f"seed={seed}",
    )
    chain_results[seed] = samples
    chain_step_size_traces[seed] = step_sizes

    gc.collect()
    jax.clear_caches()

print()
print("======================================")
print("ZUSAMMENFASSUNG MULTI-CHAIN-PRODUKTIONSLAUF (pSMILE)")
print("======================================")
for seed, samples in chain_results.items():
    status = "FEHLGESCHLAGEN (NaN/Inf)" if samples is None else f"OK, {samples.shape[0]} Samples"
    print(f"  seed={seed}: {status}")
    if chain_step_size_traces[seed]:
        trace = chain_step_size_traces[seed]
        print(f"    mean(step_size) Verlauf: {trace[0]:.6g} -> {trace[-1]:.6g}")

all_ok = all(v is not None for v in chain_results.values())
print()
if all_ok:
    print("Alle Chains stabil.")
else:
    print("Mindestens eine Chain ist fehlgeschlagen -- initial_step_size ggf. anpassen.")


Fester Anfangs-step_size fuer diesen Lauf: 0.0005
Chains: [1, 2, 3]   N_SAMPLES_PER_CHAIN=90   THINNING_STEPS=30

=== Chain seed=1 ===


[seed=1] Sample 1/90: endlich, Distanz vom Start=0.0137, mean(step_size)=0.000609497, Dauer=293.3s


[seed=1] Sample 2/90: endlich, Distanz vom Start=0.0340, mean(step_size)=0.000714123, Dauer=2.0s


[seed=1] Sample 3/90: endlich, Distanz vom Start=0.0609, mean(step_size)=0.000757833, Dauer=2.0s


[seed=1] Sample 4/90: endlich, Distanz vom Start=0.0966, mean(step_size)=0.000820302, Dauer=2.0s


...

[seed=3] Sample 87/90: endlich, Distanz vom Start=7.7741, mean(step_size)=0.000455226, Dauer=2.0s


[seed=3] Sample 88/90: endlich, Distanz vom Start=7.7545, mean(step_size)=0.000455044, Dauer=2.0s


[seed=3] Sample 89/90: endlich, Distanz vom Start=7.7528, mean(step_size)=0.000445764, Dauer=2.0s


[seed=3] Sample 90/90: endlich, Distanz vom Start=7.7613, mean(step_size)=0.000445764, Dauer=2.0s



ZUSAMMENFASSUNG MULTI-CHAIN-PRODUKTIONSLAUF (pSMILE)
  seed=1: OK, 90 Samples
    mean(step_size) Verlauf: 0.000609497 -> 0.00150621
  seed=2: OK, 90 Samples
    mean(step_size) Verlauf: 0.000434063 -> 0.000908945
  seed=3: OK, 90 Samples
    mean(step_size) Verlauf: 0.00045196 -> 0.000445764

Alle Chains stabil.


In [12]:
# ============================================================
# CHAINS KOMBINIEREN (mit Burn-in-Verwerfung) + SPEICHERN
# ============================================================

assert all(v is not None for v in chain_results.values()), (
    "Mindestens eine Chain ist fehlgeschlagen (NaN/Inf) -- vor dem Speichern beheben."
)

kept_samples = []
for seed in CHAIN_SEEDS:
    samples = chain_results[seed]
    kept = samples[BURN_IN_SAMPLES:]
    print(
        f"Chain seed={seed}: {samples.shape[0]} Samples total, "
        f"{kept.shape[0]} nach Burn-in (verworfen: {BURN_IN_SAMPLES})."
    )
    assert kept.shape[0] > 0, (
        f"BURN_IN_SAMPLES ({BURN_IN_SAMPLES}) >= N_SAMPLES_PER_CHAIN ({N_SAMPLES_PER_CHAIN}) "
        f"fuer seed={seed} -- nichts uebrig zum Kombinieren."
    )
    kept_samples.append(kept)

combined_samples = np.concatenate(kept_samples, axis=0)
print("Kombinierte Samples (alle Chains, nach Burn-in):", combined_samples.shape)

non_finite_mask = ~np.isfinite(combined_samples).all(axis=1)
n_non_finite = int(non_finite_mask.sum())
if n_non_finite > 0:
    print(f"WARNUNG: {n_non_finite} Samples enthalten NaN/Inf -- werden vor dem Speichern entfernt "
          "(gleiche Behandlung wie beim frueheren step_size=0.01-Lauf).")
    combined_samples = combined_samples[~non_finite_mask]
    print("Verbleibende Samples nach Filterung:", combined_samples.shape)

assert np.isfinite(combined_samples).all(), (
    "Kombinierte Samples enthalten immer noch NaN/Inf nach Filterung -- Abbruch vor dem Speichern."
)

RESULT_DIR.mkdir(parents=True, exist_ok=True)
samples_path = RESULT_DIR / f"{RUN_NAME}_samples.npy"
np.save(samples_path, combined_samples)
print("Samples gespeichert:", samples_path)


Chain seed=1: 90 Samples total, 70 nach Burn-in (verworfen: 20).
Chain seed=2: 90 Samples total, 70 nach Burn-in (verworfen: 20).
Chain seed=3: 90 Samples total, 70 nach Burn-in (verworfen: 20).


Kombinierte Samples (alle Chains, nach Burn-in): (210, 540672)


Samples gespeichert: /dss/dsshome1/00/ra58vit2/Masterarbeit/method_results/mile_qwen_agnews/mile_psmile_k3_prior0.02_step0.0005_mb32_spc90_thin30_valeval_samples.npy


In [13]:
# ============================================================
# AUSWERTUNG AUF EINER ZUFALLS-TEILMENGE DES VALIDATION-SETS
# (EVAL_MODE, Default "val") -- NICHT das Testset (Test-Contamination
# vermeiden, siehe Intro-Zelle) UND NICHT das volle Validation-Set:
# fuer einen reinen Kandidaten-Vergleich (3 Step-Size-Kandidaten) reicht
# eine Teilmenge, das spart Laufzeit. Fixer Seed (nicht env-abhaengig),
# damit ALLE DREI Kandidaten-Notebooks exakt dieselbe Teilmenge nutzen --
# sonst waere der Vergleich zwischen Kandidaten durch unterschiedliche
# Beispiele konfundiert. Fuer den spaeter final gewaehlten Kandidaten
# wird separat nochmal auf dem VOLLEN Testset ausgewertet.
# ============================================================

import json

x_eval_full, y_eval_full = data.get(EVAL_MODE)
n_eval_full = int(y_eval_full.shape[0])
print(f"Eval-Split '{EVAL_MODE}' Beispiele (gesamt):", n_eval_full)

EVAL_N_SUBSET = int(os.environ.get("EVAL_N_SUBSET", "3000"))
EVAL_SUBSET_SEED = 20260922  # fest, NICHT aus env -- muss ueber alle 3 Kandidaten identisch sein.

if 0 < EVAL_N_SUBSET < n_eval_full:
    subset_rng = np.random.RandomState(EVAL_SUBSET_SEED)
    subset_indices_host = np.sort(
        subset_rng.choice(n_eval_full, size=EVAL_N_SUBSET, replace=False)
    )
    subset_indices = jnp.asarray(subset_indices_host)
    x_eval = jax.tree.map(lambda arr: arr[subset_indices], x_eval_full)
    y_eval = y_eval_full[subset_indices]
    print(
        f"Zufalls-Teilmenge fuer Kandidaten-Vergleich: {EVAL_N_SUBSET} von {n_eval_full} "
        f"Beispielen (Seed={EVAL_SUBSET_SEED}, identisch ueber alle Kandidaten-Notebooks)."
    )
else:
    x_eval, y_eval = x_eval_full, y_eval_full
    print("EVAL_N_SUBSET <= 0 oder >= Gesamtgroesse -- volles Eval-Set wird verwendet.")

print("Eval-Beispiele fuer diesen Lauf:", int(y_eval.shape[0]))

eval_key = random.PRNGKey(777)
eval_probs, eval_mean_probs, eval_key = qpu.compute_posterior_predictive_probabilities(
    env=env, rebuild_full_params=rebuild_full_params,
    sample_thetas=combined_samples, x_eval=x_eval, y_eval=y_eval,
    rng_key=eval_key, eval_batch_size=EVAL_BATCH_SIZE,
)

probabilities_path = RESULT_DIR / f"{RUN_NAME}_probabilities.npy"
np.save(probabilities_path, eval_probs)
print(f"{EVAL_MODE}-Wahrscheinlichkeiten gespeichert:", probabilities_path)

eval_metrics = qpu.compute_predictive_metrics(
    sample_probabilities=eval_probs, mean_probabilities=eval_mean_probs, y_true=y_eval,
)


def _scalar(d, key):
    if key not in d:
        return None
    value = d[key]
    return float(value.mean()) if hasattr(value, "mean") else float(value)


accuracy = _scalar(eval_metrics, "accuracy")
nll = _scalar(eval_metrics, "posterior_predictive_nll")
lppd = -nll if nll is not None else None
brier = _scalar(eval_metrics, "brier_score")
mean_mi = _scalar(eval_metrics, "mutual_information")
mean_pred_entropy = _scalar(eval_metrics, "predictive_entropy")
mean_exp_entropy = _scalar(eval_metrics, "expected_entropy")

print("Accuracy:", accuracy)
print("LPPD:", lppd)
print("Posterior predictive NLL:", nll)
print("Brier Score:", brier)
print("Mean predictive entropy:", mean_pred_entropy)
print("Mean expected entropy:", mean_exp_entropy)
print("Mean mutual information:", mean_mi)

summary = {
    "accuracy": accuracy,
    "lppd": lppd,
    "posterior_predictive_nll": nll,
    "brier_score": brier,
    "mean_predictive_entropy": mean_pred_entropy,
    "mean_expected_entropy": mean_exp_entropy,
    "mean_mutual_information": mean_mi,
    "run_name": RUN_NAME,
    "method": "pSMILE (NOISYSMILE, Mini-Batch) -- step-size candidate run",
    "dataset": "AG News",
    "model": "Qwen2.5-0.5B",
    "chain_seeds": CHAIN_SEEDS,
    "prior_std": PRIOR_STD,
    "minibatch_size": MINIBATCH_SIZE,
    "initial_step_size": INITIAL_STEP_SIZE,
    "n_samples_per_chain_raw": N_SAMPLES_PER_CHAIN,
    "burn_in_samples": BURN_IN_SAMPLES,
    "n_samples_kept_total": int(combined_samples.shape[0]),
    "thinning_steps": THINNING_STEPS,
    "eval_mode": EVAL_MODE,
    "eval_n_subset": EVAL_N_SUBSET,
    "eval_n_full": n_eval_full,
    "eval_subset_seed": EVAL_SUBSET_SEED,
    "n_eval_examples": int(y_eval.shape[0]),
    "n_parameter_dimensions": int(combined_samples.shape[1]),
    "all_samples_finite": bool(np.isfinite(combined_samples).all()),
    "step_size_traces_mean_first_last": {
        str(seed): [chain_step_size_traces[seed][0], chain_step_size_traces[seed][-1]]
        for seed in CHAIN_SEEDS
    },
}

summary_path = RESULT_DIR / f"{RUN_NAME}_summary.json"
with open(summary_path, "w", encoding="utf-8") as f:
    json.dump(summary, f, indent=2)
print("Summary gespeichert:", summary_path)


Eval-Split 'val' Beispiele (gesamt): 12000


Zufalls-Teilmenge fuer Kandidaten-Vergleich: 3000 von 12000 Beispielen (Seed=20260922, identisch ueber alle Kandidaten-Notebooks).
Eval-Beispiele fuer diesen Lauf: 3000


Evaluation examples: 3000
Evaluation batch size: 64
Posterior samples: 210
Evaluating posterior sample 1/210


Evaluating posterior sample 2/210


Evaluating posterior sample 3/210


...

Evaluating posterior sample 209/210


Evaluating posterior sample 210/210



Evaluation completed
Probability tensor shape: (210, 3000, 4)
Expected shape: (210, 3000, 4)
All probabilities finite: True
val-Wahrscheinlichkeiten gespeichert: /dss/dsshome1/00/ra58vit2/Masterarbeit/method_results/mile_qwen_agnews/mile_psmile_k3_prior0.02_step0.0005_mb32_spc90_thin30_valeval_probabilities.npy


Accuracy: 0.9123333096504211
LPPD: -0.24900373816490173
Posterior predictive NLL: 0.24900373816490173
Brier Score: 0.12961220741271973
Mean predictive entropy: 0.3532269597053528
Mean expected entropy: 0.3158271312713623
Mean mutual information: 0.03739982470870018
Accuracy: 0.9123333096504211
LPPD: -0.24900373816490173
Posterior predictive NLL: 0.24900373816490173
Brier Score: 0.12961220741271973
Mean predictive entropy: 0.3532269597053528
Mean expected entropy: 0.3158271312713623
Mean mutual information: 0.03739982470870018


Summary gespeichert: /dss/dsshome1/00/ra58vit2/Masterarbeit/method_results/mile_qwen_agnews/mile_psmile_k3_prior0.02_step0.0005_mb32_spc90_thin30_valeval_summary.json
